# 16.4 KV cache 怎麼縮小？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：舊筆記不用重寫：只加新字的K/V**

cache保存每層筆記；不是提前知道下一字答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/cache.svg")))

**先想一想：**四個Q head、一個KV head，cache能縮幾倍？

多位Query讀者共用較少Key/Value記憶筆記，是GQA的直覺。cache可變小，但投影結構改變，不是原權重任意reshape就有相同品質。

**把直覺寫成數學：**K/V heads=Hkv，Q heads=Hq；Hq必須整除Hkv，本實作存未擴張cache。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

ids = torch.tensor([[1, 2, 3]])
for kv in (4, 1):
    model = TinyLM(ModelConfig(width=16, heads=4, kv_heads=kv))
    cache = model(ids)["cache"]
    print(kv, sum(t.numel() * t.element_size() for pair in cache for t in pair), "cache bytes")

**如何讀結果：**cache縮小不保證任務品質或wall time改善；另訓練匹配架構。

**只改一件事：**只改prefix長度，確認cache bytes線性變化。
